In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

hasil = sentiment_analyzer("Gamesnya bagus sekali, sangat memuaskan!")
print(hasil)


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/808k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[{'label': 'positive', 'score': 0.9923426508903503}]


In [ ]:
import pandas as pd

df = pd.read_csv('/content/dataset_petualangan_jokowi.csv')   # dari Tugas 2
sample = df.head(10).copy()

hasil = sentiment_analyzer(sample['teks'].astype(str).tolist())
sample['prediksi_bert'] = [h['label'] for h in hasil]
sample['confidence']    = [round(h['score'], 3) for h in hasil]

print(sample[['teks', 'sentimen', 'prediksi_bert', 'confidence']])


                                                teks sentimen prediksi_bert  \
0                                         woww keren  positif      positive   
1  gimana ya? game nya bagus sih standar tapi,leb...   netral      negative   
2                                    game nya bagus🥰  positif      positive   
3  kalo ketemu VM jangan di lawan lari aja nanti ...   netral      negative   
4                                   mana jokowinya??   netral      negative   
5  masih banyak banget bug nya tpi gameplay nya seru   netral      positive   
6                                       gacor Jokowi   netral       neutral   
7  Karena saat saya keluar dari game karakter nya...  negatif      negative   
8                             kurang mas Anies doang  negatif      negative   
9                                              bagus  positif      positive   

   confidence  
0       0.759  
1       0.991  
2       0.997  
3       0.990  
4       0.983  
5       0.997  
6       0.501  
7 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

X = df['teks'].astype(str)
y = df['sentimen']
_, X_test_raw, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

hasil_bert = sentiment_analyzer(X_test_raw.tolist(), truncation=True, max_length=512)
pred_bert  = [h['label'] for h in hasil_bert]

mapping = {'positive': 'positif', 'negative': 'negatif', 'neutral': 'netral'}
pred_bert_map = [mapping.get(p.lower(), p) for p in pred_bert]

print('=== DistilBERT / RoBERTa Indonesia ===')
print(f'Akurasi: {accuracy_score(y_test, pred_bert_map):.4f}')
print('\n', classification_report(y_test, pred_bert_map))

=== DistilBERT / RoBERTa Indonesia ===
Akurasi: 0.5173

               precision    recall  f1-score   support

     negatif       0.19      0.83      0.31        42
      netral       0.92      0.26      0.40       217
     positif       0.72      0.76      0.74       174

    accuracy                           0.52       433
   macro avg       0.61      0.62      0.48       433
weighted avg       0.77      0.52      0.53       433

